# ToolStrategy详解
ToolStrategy通过 工具调用 （Tool Calling/Function Calling）实现结构化输出，所以LangChain会在消息列表末尾 追加一条ToolMessage ，让整个链路完整。但实际上没有实际的工具执行，这是一条伪消息。

ToolStrategy的配置包含三个主要参数：
```python
class ToolStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    tool_message_content: str | None
    handle_errors: Union[bool, str, type[Exception], tuple[type[Exception],
...], Callable[[Exception], str]]
```

- schema（必需参数）：与提供商策略的schema参数功能一致，支持 `Pydantic模型` 、`TypedDict` 、 `JSON Schema` 、 `数据类(@dataclass)` ，同时还支持联合类型 `Union[类型1, 类型2]` （允许模型根据输入内容选择最匹配的数据结构）。
  
- tool_message_content（可选参数）：用于自定义生成结构化输出时，会话历史中记录的提示信息。默认使用展示输出数据的标准响应语句。
  
- handle_errors（可选参数）：用于指定数据校验失败时的重试策略， 默认值为True 。

In [1]:
from langchain.agents import create_agent
from dotenv import load_dotenv
import os
from langchain_qwq import ChatQwen
from rich import print as rprint
from langchain.tools import tool

load_dotenv(override=True)
model = ChatQwen(
    model="deepseek-v4-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
)

##  结构化输出：schema参数

### 输出模式：Pydantic类型
Pydantic类型的Schema支持数据验证，是优先推荐使用的方式。

In [2]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContactInfo)
)
response = agent.invoke({
    "messages": [
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
    ]
})

rprint(response)
for msg in response["messages"]:
    msg.pretty_print()
# print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='aa6f9b55-eaea-4a5c-84fe-4aec40682555'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 
'用户要求从一段话中提取结构化信息，包括姓名、邮箱和手机号。这段话中提到了"小明"的邮箱和手机号。我需要使用ContactInf
o工具来提取这些信息。\n\n让我解析一下：\n- 姓名：小明\n- 邮箱：songhk@atguigu.com\n- 手机号：12345678912'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 157,
                    'prompt_tokens': 344,
                    'total_tokens': 501,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 71,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'dashscope',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-2ae9e54e-cf14-96a5-8837-3288bad2d2c1',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f1d2c-e341-7e72-9448-d9ea6debf964-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_de16cbc5546641af829832c1',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 344,
                'output_tokens': 157,
                'total_tokens': 501,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 71}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='songhk@atguigu.com' phone='12345678912'",
            name='ContactInfo',
            id='6f2e5c70-c689-4a2f-b931-8c530cb04725',
            tool_call_id='call_de16cbc5546641af829832c1'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}

================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_de16cbc5546641af829832c1)
 Call ID: call_de16cbc5546641af829832c1
  Args:
    name: 小明
    email: songhk@atguigu.com
    phone: 12345678912
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='小明' email='songhk@atguigu.com' phone='12345678912'


In [3]:
# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"



In [4]:
@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"
    
    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"

In [ ]:
from typing import Literal

# 定义Pydantic Schema
class CustomerAnalysis(BaseModel):
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] =Field("潜在客户",description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None,description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")

In [6]:
from langchain_core.messages import SystemMessage
# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
    "请分析指定客户的情况："
    "1. 先搜索客户数据库了解最新情况 "
    "2. 如果是VIP客户，则发送感谢邮件 "
    "3. 基于搜索结果生成结构化分析报告 "
    "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
"messages": [{"role": "user", "content": "请分析客户张三"}]
# "messages": [{"role": "user","content": "请分析客户李四"}]
# "messages": [{"role": "user","content": "请分析客户王五"}]
# "messages": [{"role": "user","content": "今天天气如何"}]
})
# 处理结果
rprint(result)
if "structured_response" in result:
    analysis = result["structured_response"]
    print(analysis)

{
    'messages': [
        HumanMessage(
            content='请分析客户张三',
            additional_kwargs={},
            response_metadata={},
            id='fc18f529-75b2-43f9-8b4e-7187af693988'
        ),
        AIMessage(
            content='好的，我先来搜索客户张三的信息',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 
'好的，我需要按照用户的要求来分析客户张三的情况。首先，我需要搜索客户数据库了解最新情况。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 77,
                    'prompt_tokens': 600,
                    'total_tokens': 677,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 21,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'dashscope',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-3d16f900-ff15-9681-881b-9712af43395e',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f1d3b-9c89-7823-8bd5-97babf1b4937-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '张三'},
                    'id': 'call_8bd6b2effd384fa49387fc7f',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 600,
                'output_tokens': 77,
                'total_tokens': 677,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 21}
            }
        ),
        ToolMessage(
            content='客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000',
            name='search_customer_database',
            id='4178d0fe-0a33-47d8-be03-af66d273a9b2',
            tool_call_id='call_8bd6b2effd384fa49387fc7f'
        ),
        AIMessage(
            content='找到了张三的信息！他是 **VIP客户**。现在按流程，我先发送感谢邮件，同时生成分析报告。\n\n',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': '找到了张三的信息，他是VIP客户。根据要求：\n\n1. ✅ 搜索客户数据库了解最新情况
- 已完成\n2. 如果是VIP客户，则发送感谢邮件 - 张三确实是VIP客户，所以需要发送感谢邮件\n3. 
基于搜索结果生成结构化分析报告 - 需要生成报告\n4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象 - 
找到了客户信息，所以不需要返回空对象\n\n所以接下来我需要：\n1. 发送感谢邮件给张三\n2. 
生成结构化分析报告\n\n先发送邮件，然后生成报告。这两步没有依赖关系，可以同时进行。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 315,
                    'prompt_tokens': 693,
                    'total_tokens': 1008,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 134,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 512}
                },
                'model_provider': 'dashscope',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-0bb859f4-0c6c-9577-acb2-7a2bc868002a',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f1d3b-a3d0-7a11-b1a6-6bf6c2271675-0',
            tool_calls=[
                {
                    'name': 'send_email',
                    'args': {'customer': '张三'},
                    'id': 'call_116a20bbf0714af295d79a09',
                    'type': 'tool_call'
                },
                {
                    'name': 'CustomerAnalysis',
           

customer_name='张三' customer_tier='VIP客户' recent_activity='最近购买日期：2026-01-15' spending_level='高' send_email=True
